In [2]:
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math as py_math
from kan import KAN
import scipy
import matplotlib.pyplot as plt
from datetime import date
from torch.utils.tensorboard import SummaryWriter
%load_ext tensorboard
%rm -rf ./log/
%mkdir log
%tensorboard --logdir=log/ --host localhost --port 6007


DATA_PATH = "Data/cylinder_nektar_wake.mat"

In [4]:
def load_synthetic_data(file_path, N_train=10000):
    try:
        data = scipy.io.loadmat(file_path)
    except FileNotFoundError:
        raise FileNotFoundError(f"No file {file_path} found.")

    # 2. Extract Raw Arrays
    # Assuming standard shapes: X_star (N, 2), t (T, 1), U_star (N, 2, T), P_star (N, T)
    t_star = data['t'].flatten()[:, None]  
    X_star = data['X_star']                
    U_star = data['U_star']                
    P_star = data['p_star']                

    N_spatial = X_star.shape[0]
    N_time = t_star.shape[0]

    # 3. Spatiotemporal Flattening (Meshgrid analog)
    # Tile the spatial coordinates for every time step
    x_flat = np.tile(X_star[:, 0:1], (1, N_time)).flatten()[:, None]
    y_flat = np.tile(X_star[:, 1:2], (1, N_time)).flatten()[:, None]
    
    # Repeat the time coordinates for every spatial point
    t_flat = np.repeat(t_star, N_spatial, axis=0)

    # Flatten the velocity and pressure fields correspondingly
    u_flat = U_star[:, 0, :].flatten()[:, None]
    v_flat = U_star[:, 1, :].flatten()[:, None]
    p_flat = P_star.flatten()[:, None]

    # 4. Stochastic Subsampling
    # We select N_train points randomly to serve as our empirical supervision anchors
    total_points = x_flat.shape[0]
    idx = np.random.choice(total_points, N_train, replace=False)

    # 5. Conversion to PyTorch Tensors
    to_tensor = lambda arr: torch.tensor(arr[idx, :], dtype=torch.float32)
    
    training_tensors = {
        'x': to_tensor(x_flat),
        'y': to_tensor(y_flat),
        't': to_tensor(t_flat),
        'u': to_tensor(u_flat),
        'v': to_tensor(v_flat),
        'p': to_tensor(p_flat)
    }
    
    print(f"Data parsed successfully. Subsampled {N_train} points from a total of {total_points}.")
    return training_tensors


training_data = load_synthetic_data(DATA_PATH)

Data parsed successfully. Subsampled 10000 points from a total of 1000000.


In [5]:
def ns_residual(model, x, y, t, nu=0.01):
    """
    Evaluates the physical constraint violations (residuals) of the governing PDEs.
    """
    x.requires_grad_(True)
    y.requires_grad_(True)
    t.requires_grad_(True)
    
    u, v, p = model(x, y, t)
    
    # First-order derivatives
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, torch.ones_like(u), create_graph=True)[0]
    
    v_t = torch.autograd.grad(v, t, torch.ones_like(v), create_graph=True)[0]
    v_x = torch.autograd.grad(v, x, torch.ones_like(v), create_graph=True)[0]
    v_y = torch.autograd.grad(v, y, torch.ones_like(v), create_graph=True)[0]
    
    p_x = torch.autograd.grad(p, x, torch.ones_like(p), create_graph=True)[0]
    p_y = torch.autograd.grad(p, y, torch.ones_like(p), create_graph=True)[0]
    
    # Second-order derivatives (Kinematic viscosity diffusion)
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, torch.ones_like(u_y), create_graph=True)[0]
    v_xx = torch.autograd.grad(v_x, x, torch.ones_like(v_x), create_graph=True)[0]
    v_yy = torch.autograd.grad(v_y, y, torch.ones_like(v_y), create_graph=True)[0]
    
    # Governing Equations
    f_u = u_t + u * u_x + v * u_y + p_x - nu * (u_xx + u_yy) # x-momentum
    f_v = v_t + u * v_x + v * v_y + p_y - nu * (v_xx + v_yy) # y-momentum
    f_c = u_x + v_y                                          # continuity
    
    return f_u, f_v, f_c

In [6]:
model_kan = KAN(width=[2, 20, 20, 1], grid=5, k=3)

checkpoint directory created: ./model
saving model version 0.0
